# Day 30 — Practice SOLUTIONS
### A/B Test Design & Analysis · Module 2 · Topic 2.19

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

| Part | What it tests | Count |
|---|---|---|
| **A** | Predict the output | 8 |
| **B** | Spot & fix the bug | 4 |
| **C** | Write the code | 5 |
| **D** | Challenge | 2 |

**How to work:** for Part A, write your answer down *before* running anything.

Three questions to keep asking yourself:

- **Did I size it before I ran it?**
- **Did I check the split?**
- **Does the decision come from the interval, or from the p-value?**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.simplefilter("ignore")

def n_per_arm(p0, mde, power=0.80, alpha=0.05):
    p1   = p0 + mde
    pbar = (p0 + p1) / 2
    za, zb = stats.norm.ppf(1 - alpha/2), stats.norm.ppf(power)
    return int(np.ceil((za*np.sqrt(2*pbar*(1-pbar)) +
                        zb*np.sqrt(p0*(1-p0) + p1*(1-p1)))**2 / mde**2))

def two_prop(c_a, n_a, c_b, n_b, alpha=0.05):
    p_a, p_b = c_a/n_a, c_b/n_b
    diff = p_b - p_a
    pbar = (c_a + c_b)/(n_a + n_b)
    z = diff / np.sqrt(pbar*(1-pbar)*(1/n_a + 1/n_b))
    p = 2*(1 - stats.norm.cdf(abs(z)))
    se = np.sqrt(p_a*(1-p_a)/n_a + p_b*(1-p_b)/n_b)
    c  = stats.norm.ppf(1 - alpha/2)
    return diff, p, (diff - c*se, diff + c*se)

print("ready")

---
# Part A — Predict the output

Eight short cells. Write your prediction first, then run it.

### A1 · How much does a smaller baseline cost you?

In [ ]:
print(n_per_arm(0.05, 0.01))      # 5% baseline, catch +1pp
print(n_per_arm(0.05, 0.005))     # 5% baseline, catch +0.5pp
print(n_per_arm(0.20, 0.01))      # 20% baseline, catch +1pp

**8,158 · 31,234 · 25,583.**

Two things to read off this:

- **Halving the MDE roughly quadruples n** (8,158 → 31,234 is 3.8×). That is the quadrupling
  rule, and it holds at any baseline.
- **A higher baseline costs more for the same absolute lift.** A 20% rate needs 25,583 to
  catch +1pp where a 5% rate needs 8,158, because the variance of a proportion
  `p(1−p)` is largest near 50%.

The practical consequence: +1pp is a *much* bigger ask on a 5% metric (a 20% relative lift)
than on a 20% metric (a 5% relative lift), and the sample sizes reflect that in the opposite
direction from most people's intuition.

### A2 · The test and the interval disagree about which SE to use

In [ ]:
c_a, n_a, c_b, n_b = 250, 5000, 900, 5000
p_a, p_b = c_a/n_a, c_b/n_b
pbar = (c_a + c_b)/(n_a + n_b)

pooled   = np.sqrt(pbar*(1-pbar)*(1/n_a + 1/n_b))
unpooled = np.sqrt(p_a*(1-p_a)/n_a + p_b*(1-p_b)/n_b)
print(f"pooled   {pooled:.6f}")
print(f"unpooled {unpooled:.6f}")
print(f"ratio    {pooled/unpooled:.3f}")

**0.006380, 0.006247, ratio 1.021.**

They differ by about 2% here, and they are answering different questions:

- The **pooled** SE assumes the null hypothesis is true, so both arms share one underlying
  rate and you estimate it from all the data at once. That is the right assumption *for the
  test*, because the test asks "how surprising is this under H₀?"
- The **unpooled** SE makes no such assumption. That is the right one *for the interval*,
  because the interval is asking "what range of true differences is compatible with this?"

Use the pooled one for z and p, and the unpooled one for the CI. When the two rates are
close the difference is invisible; when they are far apart (5% vs 18%, as here) it is not.

### A3 · Significant?

In [ ]:
diff, p, (lo, hi) = two_prop(400, 10_000, 450, 10_000)
print(f"lift {diff*100:+.3f}pp   p = {p:.5f}")
print(f"95% CI [{lo*100:+.3f}, {hi*100:+.3f}]pp")
print(f"relative {diff/0.04:+.1%}")

**+0.500pp, p = 0.0797, CI [−0.059, +1.059]pp, relative +12.5%.**

Not significant — and look how easy it would be to write this up as a win. "Conversion rose
12.5%" is true. The interval runs from a small loss to a lift twice the size of the point
estimate.

This is the n = 6,000 case from the slides: a worthwhile effect is entirely compatible with
the data, and so is nothing. The honest sentence is *"inconclusive; we would need a larger
test"*, not *"no effect"* and certainly not *"+12.5%"*.

### A4 · Is this split acceptable?

In [ ]:
res = stats.chisquare([20_300, 19_700], [20_000, 20_000])
print(f"split {20300/40000:.3%} / {19700/40000:.3%}")
print(f"chi2 = {res.statistic:.2f}   p = {res.pvalue:.4f}")

**50.75% / 49.25%, chi² = 9.00, p = 0.0027.**

It looks tiny — three-quarters of a percentage point off — and it is a **sample ratio
mismatch**. On 40,000 users a 50/50 randomiser should not produce a gap of 600 users; the
probability of being this far off by chance is about 1 in 370.

⚠️ Do not analyse this test. Something dropped users unevenly, and whatever it dropped was
not random, so the arms are no longer comparable. Find the bug and rerun.

The point of the exercise: your eye cannot judge an SRM. 50.75/49.25 looks fine and is not.
The chi-square takes one line.

### A5 · How many looks before you are fooling yourself?

In [ ]:
for k in [1, 3, 5, 10]:
    print(f"{k:>2} independent tests -> {1 - 0.95**k:.1%} chance of at least one 'win'")

**5.0% · 14.3% · 22.6% · 40.1%.**

This is the formula behind the peeking simulation, the segment simulation, and Day 28's
multiplicity slide — the same arithmetic in three costumes.

⚠️ One caveat worth stating: `1 − 0.95^k` assumes the k tests are **independent**. Successive
peeks at a growing dataset are *not* independent — each peek contains all the earlier data —
which is why the measured peeking rate (25% at 20 looks) is *lower* than `1 − 0.95^20` = 64%.
Correlated looks are less damaging than independent ones, but still far worse than one look.

### A6 · Reading an interval

In [ ]:
diff, p, (lo, hi) = two_prop(1000, 10_000, 1100, 10_000)
print(f"absolute {diff*100:+.3f}pp   95% CI [{lo*100:+.3f}, {hi*100:+.3f}]pp")
print(f"relative {diff/0.10:+.1%}     95% CI [{lo/0.10:+.1%}, {hi/0.10:+.1%}]")
print(f"p = {p:.5f}")

**+1.000pp, CI [+0.15, +1.85]pp, relative +10.0% with CI [+1.5%, +18.5%], p = 0.021.**

Significant, and notice how wide the relative interval is: the data is compatible with
anything from a 1.5% lift to an 18.5% lift. A stakeholder told only "+10%" will plan as if
10% is the answer.

Converting the interval to relative terms by dividing both ends by the control rate is a
good-enough approximation when the control rate is well estimated, which it is at n = 10,000.

### A7 · The cost of precision

In [ ]:
print(n_per_arm(0.10, 0.02))
print(n_per_arm(0.10, 0.04))
print(round(n_per_arm(0.10, 0.02) / n_per_arm(0.10, 0.04), 2))

**3,841 · 1,035 · 3.71.**

Halving the MDE from +4pp to +2pp multiplied the cost by 3.7 — close to the 4× the
1/MDE² relationship predicts. (It is not exactly 4 because the pooled variance term also
changes a little as p₁ moves.)

Keep this ratio in your head. When someone asks "can we also detect a half-size effect?",
the answer is "yes, for four times the traffic or four times the time".

### A8 · What power did the test actually have?

In [ ]:
def power_of(p0, p1, n, alpha=0.05):
    pbar = (p0 + p1)/2
    za = stats.norm.ppf(1 - alpha/2)
    se0 = np.sqrt(2*pbar*(1-pbar)/n)
    se1 = np.sqrt((p0*(1-p0) + p1*(1-p1))/n)
    return 1 - stats.norm.cdf((za*se0 - abs(p1-p0))/se1)

print(f"n =    900/arm -> {power_of(0.10, 0.11, 900):.1%}")
print(f"n = 14,751/arm -> {power_of(0.10, 0.11, 14751):.1%}")

**10.2% and 80.0%.**

The second confirms the sample-size formula from the other direction: feed 14,751 back in
and you get the 80% you asked for.

The first is the number to remember. A test with 900 users per arm looking for a +1pp lift
on a 10% baseline has a **1 in 10** chance of finding it. If that test comes back flat, it
has told you essentially nothing — and if it comes back significant, the winner's curse says
the lift it reports is roughly three times too big.

---
# Part B — Spot & fix the bug

Each cell runs and produces a confident, wrong conclusion.

### B1 · "We hit significance on day 3, so we shipped early"

Both arms convert at exactly 10%. There is no effect.

In [ ]:
rng = np.random.default_rng(4008)
c_a = c_b = tot = 0
for day in range(1, 15):
    c_a += rng.binomial(1000, 0.10)
    c_b += rng.binomial(1000, 0.10)
    tot += 1000
    diff, p, ci = two_prop(c_a, tot, c_b, tot)
    if p < 0.05:
        print(f"Day {day}: lift {diff*100:+.3f}pp, p = {p:.4f} -> SIGNIFICANT, ship it!")
        break

**What is wrong, and what would the full test have shown?**

In [ ]:
# Run the planned 14 days instead of stopping at the first dip below 0.05
rng = np.random.default_rng(4008)
c_a = c_b = tot = 0
history = []
for day in range(1, 15):
    c_a += rng.binomial(1000, 0.10)
    c_b += rng.binomial(1000, 0.10)
    tot += 1000
    diff, p, ci = two_prop(c_a, tot, c_b, tot)
    history.append((day, diff*100, p))

print(f"{'day':>4} | {'lift':>9} | {'p':>7}")
print("-"*26)
for d, lift, p in history:
    mark = "  <-- stopped here" if d == 3 else ""
    print(f"{d:>4} | {lift:>+8.3f}pp | {p:>7.4f}{mark}")

diff, p, (lo, hi) = two_prop(c_a, tot, c_b, tot)
print(f"\nFULL 14 DAYS: lift {diff*100:+.3f}pp, p = {p:.4f}, "
      f"95% CI [{lo*100:+.3f}, {hi*100:+.3f}]pp")

**The bug: the loop stops the moment p dips below 0.05.** That is peeking, and the `break`
makes it explicit.

On day 3 the test showed **+1.600pp, p = 0.042** and the team shipped. Running the planned
14 days gives **+0.300pp, p = 0.4025, CI [−0.40, +1.00]pp** — nothing, which is the truth,
because both arms were set to exactly 10%.

The day-3 result was not a small real effect that faded. It was noise, caught at the moment
it happened to look biggest. And note that the p-value *rose* as more data arrived, which is
exactly what you expect when there is nothing there.

**The fix:** compute the end date from `n_per_arm()` before launch and do not evaluate the
primary metric before it. If you genuinely need the option to stop early, you must choose a
sequential design *in advance* — you cannot bolt one on afterwards.

### B2 · "Randomising by pageview gives us more data, so more power"

500 users, randomised by **pageview**. Both arms are identical — there is no effect.

In [ ]:
rng = np.random.default_rng(4002)
n_users  = 500
views    = rng.poisson(8, n_users) + 1
user_eff = rng.normal(0, 1.2, n_users)          # users differ a lot from each other
arm      = np.arange(n_users) % 2
rate     = 1/(1 + np.exp(-(-2.0 + user_eff)))   # SAME for both arms: no effect
clicks   = rng.binomial(views, rate)

df = pd.DataFrame({"arm": arm, "views": views, "clicks": clicks})
g = df.groupby("arm")[["views", "clicks"]].sum()
diff, p, ci = two_prop(g.clicks[0], g.views[0], g.clicks[1], g.views[1])
print(f"pageview-level: {g.clicks[0]/g.views[0]:.4f} vs {g.clicks[1]/g.views[1]:.4f}")
print(f"lift {diff*100:+.3f}pp, p = {p:.4f}  -> significant!")

**Why is this p-value wrong?**

In [ ]:
# The right unit is the USER, because that is what was randomised
user_rate = df.clicks / df.views
t = stats.ttest_ind(user_rate[df.arm == 0], user_rate[df.arm == 1], equal_var=False)
print(f"USER-level:     {user_rate[df.arm==0].mean():.4f} vs "
      f"{user_rate[df.arm==1].mean():.4f}")
print(f"Welch p = {t.pvalue:.4f}  -> not significant, which is correct")
print()
print(f"pageviews counted as independent : {df.views.sum():,}")
print(f"actually independent units       : {n_users:,}")
print(f"the test behaved as if it had {df.views.sum()/n_users:.1f}x more information than it had")

**The bug: the analysis treats 4,390 pageviews as 4,390 independent observations. They are
not — they come from 500 users, and pageviews from the same user are strongly correlated.**

- pageview-level: 16.13% vs 18.74%, **p = 0.0227** — "significant"
- user-level: **p = 0.1720** — not significant, which is the truth

The standard error was computed as if there were nearly nine times more independent
information than actually existed, so it came out roughly three times too small, and the
p-value followed.

**The fix:** analyse at the unit you randomised. Compute a per-user rate and compare those
with a Welch t-test (or use a method that accounts for clustering). And randomise by user in
the first place — otherwise the same person sees both versions, which is its own problem.

### B3 · "It was flat overall, but it worked brilliantly on desktop"

No effect in any segment — all eight are generated at 8% in both arms.

In [ ]:
rng = np.random.default_rng(4003)
segments = ["mobile", "desktop", "tablet", "new", "returning", "US", "EU", "APAC"]
rows = []
for s in segments:
    n = 1500
    a = rng.binomial(n, 0.08); b = rng.binomial(n, 0.08)
    diff, p, ci = two_prop(a, n, b, n)
    rows.append({"segment": s, "lift_pp": round(diff*100, 3), "p": round(p, 4)})

res = pd.DataFrame(rows).sort_values("p")
print(res.to_string(index=False))
print("\nDesktop is significant (p < 0.05). Recommend shipping to desktop users.")

**What is wrong, and what is the correct threshold?**

In [ ]:
k = len(segments)
bonf = 0.05 / k
best = res.iloc[0]
print(f"segments examined     : {k}")
print(f"expected 'wins' by chance: {k*0.05:.1f}")
print(f"P(at least one win)   : {1 - 0.95**k:.1%}")
print(f"Bonferroni threshold  : {bonf:.5f}")
print(f"best segment          : {best.segment}, p = {best.p}")
print(f"survives correction?  : {best.p < bonf}")
print()
print("Also note the DIRECTION: the 'significant' desktop result is NEGATIVE")
print(f"({best.lift_pp:+.3f}pp), so it is not even the win that was claimed.")

**The bug: eight segments were tested and only the winner was reported.**

With eight independent segments there is a **34%** chance of at least one reaching p < 0.05
when nothing is happening anywhere — and here one did, at p = 0.0120.

The Bonferroni threshold is 0.05/8 = **0.00625**, and 0.0120 does not clear it.

There is a second, funnier problem: the "brilliant" desktop result is **−2.400pp**. It is a
significant *loss*, not a win. Whoever wrote the recommendation sorted by p-value and never
looked at the sign — which is what happens when a segment scan is run as a search for
significance rather than as a test of a hypothesis.

**The fix:** name the segments in the plan before the test, or treat anything found
afterwards as a hypothesis for a *new* test and say how many you looked at.

### B4 · "No significant difference, so the new design doesn't work"

In [ ]:
rng = np.random.default_rng(4004)
n = 900
a = rng.binomial(n, 0.10); b = rng.binomial(n, 0.10)
diff, p, ci = two_prop(a, n, b, n)
print(f"control {a}/{n}, treatment {b}/{n}")
print(f"lift {diff*100:+.3f}pp, p = {p:.4f}")
print("Conclusion: the new design has no effect. Do not ship.")

**What is missing, and what should the conclusion be?**

In [ ]:
diff, p, (lo, hi) = two_prop(a, n, b, n)
print(f"lift {diff*100:+.3f}pp   95% CI [{lo*100:+.3f}, {hi*100:+.3f}]pp")
print(f"the interval is {(hi-lo)*100:.2f}pp wide\n")

print(f"power this test had for a +1pp lift : {power_of(0.10, 0.11, n):.1%}")
print(f"n needed for 80% power at +1pp      : {n_per_arm(0.10, 0.01):,}")
print(f"this test was {n_per_arm(0.10, 0.01)/n:.0f}x too small")

**The bug: a conclusion of "no effect" from a test with 10% power.**

The lift is +0.111pp with a 95% interval of **[−2.60, +2.82]pp** — 5.4 percentage points
wide. A +2.8pp win (a 28% relative lift, enormous) is entirely compatible with this data, and
so is a −2.6pp disaster.

The test had **10.2%** power to detect a +1pp lift. It needed **14,751** per arm for 80%
power and it had 900 — about **16× too small**.

**The correct conclusion:** *"Inconclusive. With 900 users per arm the interval runs from
−2.6pp to +2.8pp, which rules out nothing of interest. To answer this we would need roughly
14,751 per arm, about eight days of traffic."*

⚠️ "Not significant" means "we did not detect an effect". It never means "there is no
effect". The difference between those two sentences is this entire exercise.

---
# Part C — Write the code

Five tasks. Write the function, then run the test cell underneath it.

### C1 · A test plan calculator

Your checkout page converts at **4%**. You get **5,000 users per day per arm**.

Write `plan(p0, mde, per_day)` that prints the sample size, the number of days, and the
number of days **rounded up to whole weeks**. Then use it to decide whether chasing a +0.5pp
lift is realistic.

In [ ]:
def plan(p0, mde, per_day):
    # your code here
    pass

In [ ]:
def plan(p0, mde, per_day, power=0.80):
    n = n_per_arm(p0, mde, power)
    days = n / per_day
    weeks = int(np.ceil(days / 7))
    print(f"  baseline {p0:.1%}, MDE +{mde*100:g}pp (relative {mde/p0:+.0%})")
    print(f"  -> {n:,} per arm, {n*2:,} users in total")
    print(f"  -> {days:.1f} days at {per_day:,}/day/arm")
    print(f"  -> run {weeks} whole week(s) = {weeks*7} days")
    return n

for mde in [0.005, 0.01, 0.02]:
    plan(0.04, mde, 5000); print()

| MDE | n per arm | days | run for |
|---|---|---|---|
| +0.5pp (+12% rel) | 25,551 | 5.1 | 1 week |
| +1.0pp (+25% rel) | 6,745 | 1.3 | 1 week |
| +2.0pp (+50% rel) | 1,863 | 0.4 | 1 week |

**Chasing +0.5pp is entirely realistic here** — 5.1 days of traffic, which rounds to one
week anyway. With 5,000 users per day per arm you are in a comfortable position, and you
should set your MDE at +0.5pp rather than +1pp because it costs you nothing extra in
calendar time.

That is the real value of running this calculation: it told you to be *more* ambitious, not
less. The rounding to whole weeks is doing the work — three of these three options cost
exactly one week.

### C2 · The smallest effect your traffic can see

The inverse question. Write `mde_for_n(p0, n)` that finds the smallest lift detectable at
80% power with `n` users per arm. Use a bisection search over `n_per_arm`.

In [ ]:
def mde_for_n(p0, n, power=0.80, alpha=0.05):
    # your code here
    pass

In [ ]:
def mde_for_n(p0, n, power=0.80, alpha=0.05):
    """Smallest absolute lift detectable at `power` with n per arm."""
    lo, hi = 1e-6, 0.5
    for _ in range(80):                      # bisection
        mid = (lo + hi) / 2
        if n_per_arm(p0, mid, power, alpha) <= n:
            hi = mid
        else:
            lo = mid
    return hi

print(f"{'n per arm':>10} | {'smallest detectable lift':>25} | {'relative':>9}")
print("-"*52)
for n in [5_000, 20_000, 100_000]:
    m = mde_for_n(0.04, n)
    print(f"{n:>10,} | {m*100:>+24.4f}pp | {m/0.04:>8.1%}")

| n per arm | smallest detectable lift | relative |
|---|---|---|
| 5,000 | +1.17pp | +29.3% |
| 20,000 | +0.57pp | +14.2% |
| 100,000 | +0.25pp | +6.2% |

This is the version of the question you should ask when the traffic is fixed and
non-negotiable, which it usually is. Instead of *"how long must we run?"* you ask *"given
what we have, what is the smallest thing we could possibly see?"*

If the answer comes back larger than any effect you plausibly expect, **do not run the
test** — you already know it will be inconclusive. That is a decision worth making before
you spend two weeks, and it is the single most useful thing this function does.

### C3 · A guardrail checker

Write `guardrails(df)` that takes a frame with one row per metric (`metric`, `c_a`, `n_a`,
`c_b`, `n_b`) and flags any metric that got **significantly worse**, with a Bonferroni
correction for the number of guardrails.

In [ ]:
guard = pd.DataFrame([
    # up_is_good says which direction counts as an improvement for each metric
    {"metric": "checkout_conversion", "c_a": 1483, "n_a": 14900, "c_b": 1712, "n_b": 14900, "up_is_good": True},
    {"metric": "signup_rate",         "c_a": 2100, "n_a": 14900, "c_b": 2060, "n_b": 14900, "up_is_good": True},
    {"metric": "support_ticket_rate", "c_a":  180, "n_a": 14900, "c_b":  265, "n_b": 14900, "up_is_good": False},
    {"metric": "newsletter_optin",    "c_a":  900, "n_a": 14900, "c_b":  880, "n_b": 14900, "up_is_good": True},
])

def guardrails(df, alpha=0.05):
    # your code here
    pass

In [ ]:
def guardrails(df, alpha=0.05):
    k = len(df)
    thresh = alpha / k
    rows = []
    for _, r in df.iterrows():
        diff, p, (lo, hi) = two_prop(r.c_a, r.n_a, r.c_b, r.n_b)
        improved = (diff > 0) == bool(r.up_is_good)
        rows.append({"metric": r.metric,
                     "control": f"{r.c_a/r.n_a:.2%}",
                     "treatment": f"{r.c_b/r.n_b:.2%}",
                     "lift_pp": round(diff*100, 3),
                     "p": round(p, 6),
                     "flag": "" if p >= thresh
                             else "better" if improved else "*** WORSE ***"})
    print(f"{k} metrics, Bonferroni threshold {thresh:.5f}\n")
    return pd.DataFrame(rows)

print(guardrails(guard).to_string(index=False))

```
4 metrics, Bonferroni threshold 0.01250

             metric control treatment  lift_pp        p          flag
checkout_conversion   9.95%    11.49%    1.537 0.000018        better
        signup_rate  14.09%    13.83%   -0.268 0.503755
support_ticket_rate   1.21%     1.78%    0.570 0.000049 *** WORSE ***
   newsletter_optin   6.04%     5.91%   -0.134 0.624933
```

**Support tickets rose from 1.21% to 1.78%** — a 47% relative increase, p = 0.000049, which
clears the corrected threshold by two orders of magnitude.

This is why guardrails exist. The primary metric won convincingly, and shipping on that
alone would have pushed a 47% increase in support load into production. The two other
guardrails moved slightly in the wrong direction and neither is distinguishable from noise.

**The detail that makes this exercise worth doing:** a guardrail going *up* is not
automatically good. A naive implementation that flags `diff < 0` as "worse" would have
labelled the support-ticket spike as **better**, because the number went up. The direction
has to be carried per metric — which is what the `up_is_good` column is for. For
`support_ticket_rate`, up is bad — in real code you would carry the desired direction for
each metric rather than assuming more is better.

### C4 · A full pre-flight check

Write `preflight(c_a, n_a, c_b, n_b, planned_n, p0, mde)` that refuses to give a verdict
unless the test passes three checks: the split is clean, the planned sample size was
reached, and the test had at least 80% power for its stated MDE.

In [ ]:
def preflight(c_a, n_a, c_b, n_b, planned_n, p0, mde):
    # your code here
    pass

In [ ]:
def preflight(c_a, n_a, c_b, n_b, planned_n, p0, mde, alpha=0.05):
    problems = []

    tot = n_a + n_b
    srm_p = stats.chisquare([n_a, n_b], [tot/2, tot/2]).pvalue
    if srm_p < 0.01:
        problems.append(f"SRM: split is {n_a/tot:.2%}/{n_b/tot:.2%}, p = {srm_p:.1e}")

    if min(n_a, n_b) < planned_n:
        problems.append(f"stopped early: {min(n_a, n_b):,} per arm vs {planned_n:,} planned")

    pw = power_of(p0, p0 + mde, min(n_a, n_b), alpha)
    if pw < 0.80:
        problems.append(f"underpowered: {pw:.1%} for a +{mde*100:g}pp lift")

    if problems:
        print("DO NOT INTERPRET THIS TEST:")
        for p_ in problems:
            print("   -", p_)
        return False

    diff, p, (lo, hi) = two_prop(c_a, n_a, c_b, n_b)
    print(f"checks passed (SRM p = {srm_p:.2f}, power {pw:.0%})")
    print(f"lift {diff*100:+.3f}pp  95% CI [{lo*100:+.3f}, {hi*100:+.3f}]pp  p = {p:.3e}")
    return True

print("--- the good test ---")
_ = preflight(1483, 14900, 1712, 14900, planned_n=14751, p0=0.10, mde=0.01)

print("\n--- stopped early, and the split is broken ---")
_ = preflight(300, 3100, 340, 2900, planned_n=14751, p0=0.10, mde=0.01)

```
--- the good test ---
checks passed (SRM p = 1.00, power 80%)
lift +1.537pp  95% CI [+0.835, +2.239]pp  p = 1.805e-05

--- stopped early, and the split is broken ---
DO NOT INTERPRET THIS TEST:
   - SRM: split is 51.67%/48.33%, p = 9.8e-03
   - stopped early: 2,900 per arm vs 14,751 planned
   - underpowered: 23.6% for a +1pp lift
```

The structure is the point: **the checks come before the numbers, and failing any of them
suppresses the result entirely.** A function that printed the lift alongside a warning would
have the warning ignored — the lift is what people read.

All three problems in the second example are real and independent, and any one of them alone
is enough to make the result uninterpretable.

### C5 · Show the winner's curse yourself

Simulate a true **+2pp** lift on a 20% baseline. For n = 500, 2,000 and the properly powered
n, report the power and the average measured lift **among the tests that won**.

In [ ]:
# your code here

In [ ]:
TRUE_A, TRUE_B = 0.20, 0.22
proper = n_per_arm(TRUE_A, 0.02)
rng = np.random.default_rng(777)
TRIALS = 3000

print(f"properly powered n = {proper:,} per arm\n")
print(f"{'n per arm':>10} | {'power':>6} | {'avg lift among winners':>23} | inflation")
print("-"*62)
for n in [500, 2000, proper]:
    lifts, wins = [], 0
    for _ in range(TRIALS):
        a = rng.binomial(n, TRUE_A); b = rng.binomial(n, TRUE_B)
        diff, p, _ = two_prop(a, n, b, n)
        if p < 0.05 and diff > 0:
            wins += 1; lifts.append(diff*100)
    print(f"{n:>10,} | {wins/TRIALS:>5.0%}  | {np.mean(lifts):>+21.2f}pp | "
          f"{np.mean(lifts)/2.0:>6.2f}x")
print("\n(the truth is +2.00pp in every row)")

```
properly powered n = 6,510 per arm

 n per arm |  power |  avg lift among winners | inflation
--------------------------------------------------------------
       500 |   12%  |                 +6.26pp |   3.13x
     2,000 |   34%  |                 +3.37pp |   1.68x
     6,510 |   81%  |                 +2.24pp |   1.12x
```

The same pattern as the slides, on a different baseline and a different effect size, which
is the point of repeating it: **this is not a quirk of one example.**

At n = 500 the test wins 12% of the time and, when it does, reports **+6.26pp** for a lift
that is really +2.00pp. A team shipping on that number would plan for more than three times
the benefit they are going to get.

Even a properly powered test overstates slightly (1.12×), because you are still conditioning
on significance. The only complete fix is to report the result whether or not it was
significant — which is also the fix for publication bias in science, and the same arithmetic
drives both.

---
# Part D — Challenge

Two open tasks.

### D1 · Design a test from a brief

> *"Our email signup form converts at **6%**. The design team has rebuilt it and thinks it
> will do better. We get about **3,000 visitors a day** to that page. Marketing says anything
> below a **1 percentage point** lift is not worth the migration work. Can we run this?"*

Produce a complete one-page test plan. It must state:

1. the unit of randomisation, and why
2. the primary metric, and two guardrails
3. the MDE, with the reasoning
4. n per arm, total users, and the number of days — rounded to whole weeks
5. the fixed end date rule and what would make you stop early
6. the exact decision rule, written before any data exists

Then compute the sizing, and say whether you would recommend running it.

In [ ]:
# your code here

In [ ]:
p0, mde, per_day_total = 0.06, 0.01, 3000
per_day_arm = per_day_total / 2

n = n_per_arm(p0, mde)
days = n / per_day_arm
weeks = int(np.ceil(days / 7))

print(f"baseline            : {p0:.0%}")
print(f"MDE                 : +{mde*100:g}pp  (relative {mde/p0:+.0%})")
print(f"n per arm           : {n:,}")
print(f"total users         : {n*2:,}")
print(f"traffic             : {per_day_arm:,.0f}/day/arm")
print(f"days needed         : {days:.1f}")
print(f"run for             : {weeks} whole weeks = {weeks*7} days")
print()
print(f"smallest lift we could see in 2 weeks: "
      f"+{mde_for_n(p0, int(per_day_arm*14))*100:.2f}pp")
print(f"smallest lift we could see in 4 weeks: "
      f"+{mde_for_n(p0, int(per_day_arm*28))*100:.2f}pp")

```
baseline            : 6%
MDE                 : +1pp  (relative +17%)
n per arm           : 9,540
total users         : 19,080
traffic             : 1,500/day/arm
days needed         : 6.4
run for             : 1 whole weeks = 7 days
```

**Recommendation: yes, run it — one week.**

### The plan

**1 · Unit — the visitor (a cookie or user id), not the pageview.**
Someone may load the signup page several times before deciding. If we randomise per
pageview they could see both forms, which is confusing and also breaks independence.

**2 · Metric — signup conversion rate** (signups ÷ visitors to the page). Guardrails:
*email unsubscribe rate within 30 days* (a form that tricks people into signing up is worse
than no form) and *page load time* (the rebuild may be heavier).

**3 · MDE — +1pp absolute**, because marketing has already said below that the migration is
not worth doing. The statistics did not choose this number; the business did, which is the
correct order.

**4 · Size — 9,540 per arm, 19,080 total, 6.4 days → run 7 days.**

**5 · End date — fixed at 7 days from launch**, written in the calendar before the test
starts. The primary metric is not evaluated before then. Guardrails are checked daily and we
*may stop for harm* — a spike in unsubscribes or load time — but we *never stop for success*.

**6 · Decision rule, written now:**
- CI entirely above +1pp → ship
- CI entirely within ±1pp → do not ship; we have ruled out anything worth the work
- CI crosses +1pp but excludes 0 → take it to marketing with the interval; it is real but
  may not clear the bar
- CI wide and crossing everything → should not happen at this n; if it does, something broke

### One thing worth adding

Running 7 days costs the same as running 6.4, so we could lower the MDE to about **+0.95pp**
for free. Since whole weeks are mandatory anyway, we should size against the week, not the
other way round — and two weeks would get us down to **+0.67pp** if marketing later decides
smaller lifts matter.

### D2 · Audit a real-looking write-up

Here is an experiment summary of the kind you will be handed in a meeting. **There are four
separate things wrong with it.** Find all four, say what each one does to the conclusion, and
write the honest version.

> **Test: new product recommendation carousel**
>
> *We ran the new carousel against the current page for 5 days (Mon–Fri), randomising each
> pageview. 48,200 pageviews saw the control and 51,800 saw the treatment.*
>
> *Revenue per session rose from \$4.10 to \$4.38, a 6.8% increase (p = 0.041). We checked
> conversion rate, add-to-cart rate, bounce rate, session length, return visits and
> newsletter signups, and add-to-cart was also up (p = 0.03).*
>
> *The uplift was strongest in the first two days. Recommendation: ship.*

In [ ]:
# Use this cell for any calculations that support your audit

In [ ]:
# 1. the split
n_a, n_b = 48_200, 51_800
tot = n_a + n_b
srm = stats.chisquare([n_a, n_b], [tot/2, tot/2])
print("1. SPLIT")
print(f"   {n_a:,} / {n_b:,} = {n_a/tot:.2%} / {n_b/tot:.2%}")
print(f"   chi2 = {srm.statistic:,.1f}   p = {srm.pvalue:.3e}")
print()

# 2. multiplicity across the six metrics
k = 6
print("2. MULTIPLICITY")
print(f"   {k} metrics checked, Bonferroni threshold {0.05/k:.4f}")
print(f"   P(at least one false 'win') = {1 - 0.95**k:.1%}")
print(f"   add-to-cart p = 0.03 survives? {0.03 < 0.05/k}")
print(f"   revenue      p = 0.041 survives? {0.041 < 0.05/k}")

### The four problems

**1 · Sample ratio mismatch — and it is enormous.**
48,200 vs 51,800 is 48.2% / 51.8%, chi² = 129.6, **p = 5e-30**. A 50/50 randomiser does not
produce this. Something systematically dropped pageviews from the control arm — and whatever
it dropped was not random, so the two groups are not comparable.

*Effect on the conclusion:* **fatal**. Nothing downstream can be interpreted. This alone ends
the discussion; the test must be fixed and rerun.

**2 · Randomised by pageview, then analysed as if pageviews were independent.**
They come from far fewer users, and pageviews from one user are heavily correlated. The
standard error is too small and the p-value is too small — possibly by a large factor (B2
showed roughly 3×).

*Effect:* p = 0.041 is not a real 0.041. The true p is unknown and larger.

**3 · Six metrics were checked and the two that moved were reported.**
With six metrics there is a **26%** chance of at least one false positive. The Bonferroni
threshold is 0.05/6 = **0.0083**, and *neither* reported result clears it (0.041 and 0.03).

*Effect:* neither "win" survives correction.

**4 · Five days, Monday to Friday, with the lift concentrated in the first two.**
Two problems in one sentence. No weekend means the test never saw a large and systematically
different part of the traffic. And a lift that is "strongest in the first two days" is the
signature of a **novelty effect** — the thing that fades. The cumulative number reported
includes those first two days and therefore overstates the long-run effect.

*Effect:* the measured lift is probably an overestimate of what shipping would deliver, and
the test cannot speak to weekend behaviour at all.

### The honest version

> *"This test cannot be interpreted. The arms received 48.2% and 51.8% of traffic
> (chi² = 129.6, p = 5e-30), which indicates a bug in assignment or logging rather than
> chance, so the two groups are not comparable.*
>
> *Three further issues would need fixing before a rerun: randomisation and analysis should
> be at the user level rather than the pageview level; the six metrics examined require either
> a pre-specified primary metric or a multiplicity correction (under which neither reported
> result is significant); and the test should run for whole weeks, with the daily lift plotted
> separately from the cumulative lift, since the effect concentrated in the first two days is
> consistent with novelty.*
>
> *Recommendation: fix the assignment bug, re-launch for two full weeks with revenue per user
> as the single pre-specified primary metric, and size the test in advance."*

### The meta-lesson

Notice that **nobody lied**. Every number in the original is arithmetically correct. The
write-up fails not because the maths is wrong but because of what it leaves out: how the
split came out, what the unit was, how many metrics were examined, and what the shape of the
effect over time looked like.

Those four omissions are exactly the four things this session told you to report.

---
### Before you hand in

- For every test, did you **check the split first**?
- Did you report the **interval**, not just the p-value?
- Did you state **how many metrics or segments** you looked at?
- Did you work out the **power**, and say so when a flat result was underpowered?
- Did you ever write "no effect" where you meant "we did not detect an effect"?
- For Part D, did you find all four problems? Both D tasks are marked.

Solutions will be released after class.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*